# Nodule Description

In [ ]:
import pandas as pd
import json
import re

# Load your CSV
df = pd.read_csv("# result-csv")

# Change these column names if needed
location_col = "location_florence"
diameter_col = "diameter_florence"
margin_col = "margin_florence"
attenuation_col = "attenuation_florence"

# Clean diameter values like "[12.0]" -> "12.0"
def clean_diameter(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = x.replace("[", "").replace("]", "").replace("'", "").replace('"', "").strip()
    
    try:
        x = float(x)
        if x.is_integer():
            return str(int(x))   # 12.0 -> 12
        return str(x)            # 12.5 -> 12.5
    except:
        return x

df[diameter_col] = df[diameter_col].apply(clean_diameter)

# Build prompt
df["prompt"] = (
    "nodule characteristics: "
    + "location: " + df[location_col].astype(str).str.strip()
    + " longest diameter: " + df[diameter_col].astype(str).str.strip() + " mm"
    + " margin: " + df[margin_col].astype(str).str.strip()
    + " attenuation: " + df[attenuation_col].astype(str).str.strip()
    + "\n\nBased on the nodule characteristics provided, describe the nodule."
)

# Build chat-format messages
df["messages"] = df.apply(
    lambda row: [
        {
            "role": "system",
            "content": "You are a medical assistant that describes pulmonary nodules from structured attributes."
        },
        {
            "role": "user",
            "content": row["prompt"]
        }
    ],
    axis=1
)

# Save as JSONL
with open("description_llm_zero_shot.jsonl", "w", encoding="utf-8") as f:
    for msg in df["messages"]:
        f.write(json.dumps({"messages": msg}, ensure_ascii=False) + "\n")

print("description_llm_zero_shot.jsonl")
print(df[[diameter_col, "prompt"]].head())

# Followup Recommendation

In [ ]:
import pandas as pd
import json
import re

# -----------------------------
# File names
# -----------------------------
nodule_file = "# result-csv"

# -----------------------------
# Load CSVs
# -----------------------------
nod_df = pd.read_csv(nodule_file)

# -----------------------------
# Column names: change if needed
# -----------------------------
pid_col = "pid"
study_col = "study_yr"

location_col = "location_florence"
diameter_col = "diameter_florence"
margin_col = "margin_florence"
attenuation_col = "attenuation_florence"

# -----------------------------
# Clean helpers
# -----------------------------
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def clean_diameter(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"[\[\]'\"\"]", "", x).strip()
    try:
        x = float(x)
        if x.is_integer():
            return str(int(x))   # 12.0 -> 12
        return str(x)            # 12.5 -> 12.5
    except:
        return x

def to_int_binary(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()
    try:
        return int(float(x))
    except:
        return 0

# -----------------------------
# Clean nodule dataframe
# -----------------------------
nod_df[location_col] = nod_df[location_col].apply(clean_text)
nod_df[margin_col] = nod_df[margin_col].apply(clean_text)
nod_df[attenuation_col] = nod_df[attenuation_col].apply(clean_text)
nod_df[diameter_col] = nod_df[diameter_col].apply(clean_diameter)

# -----------------------------
# Build one prompt per pid + study_yr
# -----------------------------
grouped_rows = []

for (pid, study_yr), group in nod_df.groupby([pid_col, study_col], sort=False):
    group = group.reset_index(drop=True)

    nodule_blocks = []
    for i, row in group.iterrows():
        nodule_blocks.append(
            f"Nodule {i+1}:\n"
            f"location: {row[location_col]}\n"
            f"longest diameter: {row[diameter_col]} mm\n"
            f"margin: {row[margin_col]}\n"
            f"attenuation: {row[attenuation_col]}"
        )

    prompt = (
        f"The patient has {len(group)} nodules.\n\n"
        + "\n\n".join(nodule_blocks)
        + "\n\nBased on these nodule characteristics, describe the overall follow-up recommendation."
    )

    grouped_rows.append({
        pid_col: pid,
        study_col: study_yr,
        "prompt": prompt
    })

df = pd.DataFrame(grouped_rows)

# -----------------------------
# Build messages
# -----------------------------
system_msg = (
    "You are a medical assistant that determines follow-up recommendations "
    "from structured pulmonary nodule characteristics."
)

df["messages"] = df.apply(
    lambda row: [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": row["prompt"]}
    ],
    axis=1
)

# -----------------------------
# Save JSONL
# -----------------------------
output_file = "followup_llm_zero_shot.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for msgs in df["messages"]:
        f.write(json.dumps({"messages": msgs}, ensure_ascii=False) + "\n")

print(f"Saved {len(df)} examples to {output_file}")
print(df[[pid_col, study_col, "prompt"]].head(3).to_string(index=False))

# Longitudal Analysis

In [ ]:
import pandas as pd

# Define column names
pid_col = "pid"
study_col = "study_yr"

# Load your nodule-level data
nod_df = pd.read_csv("# result-csv")

# Count unique study years per patient (PID)
year_counts = nod_df.groupby(pid_col)[study_col].nunique()

# Filter for patients with more than 1 study year
multi_year_pids = year_counts[year_counts > 1]

# Display the results
print(f"Total unique patients: {len(year_counts)}")
print(f"Number of patients with data from multiple years: {len(multi_year_pids)}")
print(f"Total entries (rows) for these longitudinal patients: {len(nod_df[nod_df[pid_col].isin(multi_year_pids.index)])}")

# Show distribution (e.g., how many patients have 2 years vs 3 years)
print("\nDistribution of years per patient:")
print(multi_year_pids.value_counts().sort_index())

In [ ]:
import pandas as pd
import json
import re

# -----------------------------
# 1. Load and Clean Data
# -----------------------------
nodule_file = "converted_slice_test_revised.csv"
nod_df = pd.read_csv(nodule_file)

pid_col = "pid"
study_col = "study_yr"
location_col = "location_florence"
diameter_col = "diameter_florence"
margin_col = "margin_florence"
attenuation_col = "attenuation_florence"

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def clean_diameter(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"[\[\]'\"\"]", "", x).strip()
    try:
        x = float(x)
        if x.is_integer():
            return str(int(x))   # 12.0 -> 12
        return str(x)            # 12.5 -> 12.5
    except:
        return x

def to_int_binary(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()
    try:
        return int(float(x))
    except:
        return 0

# -----------------------------
# Clean nodule dataframe
# -----------------------------
nod_df[location_col] = nod_df[location_col].apply(clean_text)
nod_df[margin_col] = nod_df[margin_col].apply(clean_text)
nod_df[attenuation_col] = nod_df[attenuation_col].apply(clean_text)
nod_df[diameter_col] = nod_df[diameter_col].apply(clean_diameter)

# -----------------------------
# 2. Filter for Longitudinal Cases
# -----------------------------
# Get PIDs that appear in more than one unique study year
pids_with_history = nod_df.groupby(pid_col)[study_col].nunique()
multi_year_pids = pids_with_history[pids_with_history > 1].index
nod_df_long = nod_df[nod_df[pid_col].isin(multi_year_pids)]

# -----------------------------
# 3. Build Longitudinal Prompts
# -----------------------------
grouped_rows = []

for pid, group in nod_df_long.groupby(pid_col, sort=False):
    # Sort chronologically (Year 0 -> Year 1 -> Year 2)
    group = group.sort_values(study_col)
    unique_years = sorted(group[study_col].unique())
    
    history_text = []
    for yr in unique_years:
        year_data = group[group[study_col] == yr].reset_index(drop=True)
        nodules = []
        for i, row in year_data.iterrows():
            nodules.append(
                f"  Nodule {i+1}: {row[location_col]}, {row[diameter_col]}mm, "
                f"{row[margin_col]} margin, {row[attenuation_col]} attenuation"
            )
        history_text.append(f"Study Year {yr}:\n" + "\n".join(nodules))

    full_history = "\n\n".join(history_text)
    
    # Prompt using clinical terms for your thesis
    prompt = (
        f"The patient has been followed over {len(unique_years)} screening rounds.\n\n"
        f"{full_history}\n\n"
        "Analyze the temporal evolution and detect any significant interval changes in the nodules. "
        "Based on this longitudinal history, describe the overall follow-up recommendation and treatment procedure."
    )

    grouped_rows.append({pid_col: pid, "prompt": prompt})

df_long = pd.DataFrame(grouped_rows)

# -----------------------------
# 4. Save JSONL
# -----------------------------
system_msg = (
    "You are a medical assistant specializing in longitudinal pulmonary analysis. "
    "You determine follow-up and treatment plans by analyzing nodule progression over multiple screenings."
)

df_long["messages"] = df_long.apply(
    lambda row: [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": row["prompt"]}
    ], axis=1
)

output_file = "longitudinal_llm_zero_shot.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for msgs in df_long["messages"]:
        f.write(json.dumps({"messages": msgs}, ensure_ascii=False) + "\n")

print(f"Saved {len(df_long)} longitudinal entries to {output_file}")